# Credit Risk Final Modeling Notebook

This notebook builds the final classification models for the credit risk project.

The goal is to predict `loan_status`, where the model learns whether a loan is likely to default based on borrower, loan, and credit-history features.

The notebook is written to be presentation-friendly. Each section explains what is happening, why it matters, and how it connects to the grading requirements.


## Optional Environment Setup

Run the next cell if the notebook environment is missing the required packages.

This is included so the notebook is easier to reproduce on a clean machine or a fresh Jupyter environment.


In [1]:
# Run this cell only if your notebook environment is missing the required packages.
# It is safe to skip this cell if all imports already work.

%pip install pandas numpy matplotlib scikit-learn ipython


Note: you may need to restart the kernel to use updated packages.


## 1. Import Required Libraries

These libraries cover data handling, preprocessing, model training, evaluation, and results display.

Important note: this notebook avoids the Pandas warning caused by `select_dtypes(include='object')` by using a helper function that detects object and string columns safely.


In [2]:
import warnings
warnings.filterwarnings("default")

from pathlib import Path
import json
import numpy as np
import pandas as pd

from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC

RANDOM_STATE = 42

print("Libraries imported successfully.")


Libraries imported successfully.


## 2. Load the Dataset

The source dataset is the Kaggle Credit Risk Dataset.

The target column is `loan_status`, making this a binary classification problem.


In [3]:
DATA_PATH = Path("credit_risk_dataset.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "credit_risk_dataset.csv was not found. "
        "Make sure the notebook is running from the project folder."
    )

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())
display(df.info())


Dataset shape: (32581, 12)


,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4


<class 'pandas.DataFrame'>
RangeIndex: 32581 entries, 0 to 32580
Data columns (total 12 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   person_age                  32581 non-null  int64  
 1   person_income               32581 non-null  int64  
 2   person_home_ownership       32581 non-null  str    
 3   person_emp_length           31686 non-null  float64
 4   loan_intent                 32581 non-null  str    
 5   loan_grade                  32581 non-null  str    
 6   loan_amnt                   32581 non-null  int64  
 7   loan_int_rate               29465 non-null  float64
 8   loan_status                 32581 non-null  int64  
 9   loan_percent_income         32581 non-null  float64
 10  cb_person_default_on_file   32581 non-null  str    
 11  cb_person_cred_hist_length  32581 non-null  int64  
dtypes: float64(3), int64(5), str(4)
memory usage: 3.0 MB


None

## 3. Train / Validation / Test Split

The final rubric requires a 70% / 15% / 15% split.

A clean way to explain this during the presentation:

1. First, 70% of the data is separated for training.
2. The remaining 30% is held out from training.
3. That 30% holdout is split in half:
   - 15% validation
   - 15% final test

The validation set is used for model comparison and ensemble selection.  
The final test set is reserved for the last evaluation only.


In [4]:
target_column = "loan_status"

X = df.drop(columns=[target_column])
y = df[target_column]

# Step 1: Create the 70% training split and 30% holdout split.
X_train_raw, X_holdout_raw, y_train, y_holdout = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y
)

# Step 2: Split the 30% holdout evenly into 15% validation and 15% final test.
X_val_raw, X_test_raw, y_val, y_test = train_test_split(
    X_holdout_raw,
    y_holdout,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y_holdout
)

split_summary = pd.DataFrame({
    "Split": ["Training", "Validation", "Testing"],
    "Rows": [len(X_train_raw), len(X_val_raw), len(X_test_raw)],
    "Percent of Full Dataset": [
        len(X_train_raw) / len(df),
        len(X_val_raw) / len(df),
        len(X_test_raw) / len(df),
    ]
})

split_summary["Percent of Full Dataset"] = (split_summary["Percent of Full Dataset"] * 100).round(2)

display(split_summary)

print("Presentation explanation:")
print("The model learns on 70% of the data.")
print("The original 30% holdout is split into 15% validation and 15% final testing.")
print("This keeps final testing separate from model selection.")


,Split,Rows,Percent of Full Dataset
0,Training,22806,70.0
1,Validation,4887,15.0
2,Testing,4888,15.0


Presentation explanation:
The model learns on 70% of the data.
The original 30% holdout is split into 15% validation and 15% final testing.
This keeps final testing separate from model selection.


## 4. Warning-Safe Helper Functions

The earlier notebook produced Pandas warnings because it used:

```python
select_dtypes(include='object')
```

Newer Pandas versions are changing how object and string columns are handled.

This helper function avoids that warning by checking each column directly. Developed with the assistance of Gemini.


In [5]:
def get_text_columns(dataframe):
    """Return columns that contain object or string-like values without triggering Pandas4Warning.

    Why this exists:
    Pandas is changing string dtype behavior. The older approach
    `select_dtypes(include='object')` may still work, but it can raise a warning
    in newer environments. This helper is clearer and future-safe.
    """
    text_columns = []

    for column in dataframe.columns:
        series = dataframe[column]

        if pd.api.types.is_object_dtype(series) or pd.api.types.is_string_dtype(series):
            text_columns.append(column)

    return text_columns


def replace_blank_strings_with_nan(dataframe):
    """Convert blank strings to missing values for all text columns.

    This prevents values like '', ' ', or '   ' from being treated as real categories.
    """
    cleaned = dataframe.copy()
    text_columns = get_text_columns(cleaned)

    if text_columns:
        cleaned[text_columns] = cleaned[text_columns].replace(r"^\s*$", np.nan, regex=True)

    return cleaned


# Quick validation that the helper works without using select_dtypes(include='object').
print("Detected text columns:", get_text_columns(X_train_raw))


Detected text columns: ['person_home_ownership', 'loan_intent', 'loan_grade', 'cb_person_default_on_file']


## 5. Preprocessing Functions

The preprocessing logic is intentionally separated into two stages:

### Fit stage
Uses the training data only to learn medians, category levels, scaling ranges, and other reusable values.

### Transform stage
Applies the training-learned values to validation and test data.

This protects against data leakage.


In [6]:
GRADE_MAP = {
    "A": 1,
    "B": 2,
    "C": 3,
    "D": 4,
    "E": 5,
    "F": 6,
    "G": 7,
}


def base_clean_features(dataframe, artifacts=None, fit=False):
    """Clean raw features before encoding and scaling.

    Key grading corrections included here:
    1. `person_emp_length` is capped at 100 instead of allowing the 123 outlier to distort the feature.
    2. `loan_grade` is ordinal encoded using A=1 through G=7.
    3. `person_income` and `loan_amnt` are log-transformed to address skew.
    4. Values learned from the training data are reused for validation and test cleaning.

    Parameters
    ----------
    dataframe:
        Raw feature dataframe.
    artifacts:
        Dictionary of training-learned preprocessing values.
    fit:
        If True, learn preprocessing values from this dataframe.
        If False, reuse values learned from the training dataframe.

    Returns
    -------
    cleaned:
        Cleaned feature dataframe.
    artifacts:
        Training-learned preprocessing values.
    """

    cleaned = replace_blank_strings_with_nan(dataframe)

    if artifacts is None:
        artifacts = {}

    # Cap age to a reasonable range.
    # This avoids unrealistic ages dominating model behavior.
    cleaned["person_age"] = cleaned["person_age"].clip(lower=18, upper=100)

    # Correct grading issue:
    # Cap employment length at 100. Do not scale the 123 outlier as if it is normal.
    cleaned["person_emp_length"] = cleaned["person_emp_length"].clip(lower=0, upper=100)

    if fit:
        artifacts["person_emp_length_median"] = cleaned["person_emp_length"].median()

    cleaned["person_emp_length"] = cleaned["person_emp_length"].fillna(
        artifacts["person_emp_length_median"]
    )

    # Interest rate imputation:
    # Learn loan-grade medians from training only, then reuse them on validation/test.
    if fit:
        artifacts["loan_int_rate_by_grade"] = (
            cleaned.groupby("loan_grade")["loan_int_rate"].median().to_dict()
        )
        artifacts["loan_int_rate_overall_median"] = cleaned["loan_int_rate"].median()

    cleaned["loan_int_rate"] = cleaned["loan_int_rate"].fillna(
        cleaned["loan_grade"].map(artifacts["loan_int_rate_by_grade"])
    )
    cleaned["loan_int_rate"] = cleaned["loan_int_rate"].fillna(
        artifacts["loan_int_rate_overall_median"]
    )

    # Create engineered features before log transforms so ratios use raw financial values.
    raw_income = cleaned["person_income"].clip(lower=1)
    raw_loan_amount = cleaned["loan_amnt"].clip(lower=0)

    cleaned["debt_to_income_ratio"] = raw_loan_amount / raw_income
    cleaned["income_per_credit_year"] = raw_income / (cleaned["cb_person_cred_hist_length"] + 1)
    cleaned["credit_history_to_age_ratio"] = cleaned["cb_person_cred_hist_length"] / cleaned["person_age"]

    # Explicitly address skew in the two financial features called out in review notes.
    cleaned["person_income"] = np.log1p(raw_income)
    cleaned["loan_amnt"] = np.log1p(raw_loan_amount)

    # Replace infinite values from division and fill engineered feature gaps.
    cleaned = cleaned.replace([np.inf, -np.inf], np.nan)

    engineered_columns = [
        "debt_to_income_ratio",
        "income_per_credit_year",
        "credit_history_to_age_ratio",
    ]

    if fit:
        artifacts["engineered_feature_medians"] = {
            column: cleaned[column].median() for column in engineered_columns
        }

    for column in engineered_columns:
        cleaned[column] = cleaned[column].fillna(artifacts["engineered_feature_medians"][column])

    # Binary encoding for previous default status.
    cleaned["cb_person_default_on_file"] = (
        cleaned["cb_person_default_on_file"]
        .map({"N": 0, "Y": 1})
        .fillna(0)
        .astype(int)
    )

    # Ordinal encoding for loan grade.
    # A lower number represents a better grade; higher numbers represent higher risk.
    cleaned["loan_grade"] = cleaned["loan_grade"].map(GRADE_MAP)

    if fit:
        artifacts["loan_grade_median"] = cleaned["loan_grade"].median()

    cleaned["loan_grade"] = cleaned["loan_grade"].fillna(artifacts["loan_grade_median"])

    return cleaned, artifacts


def encode_and_scale_features(cleaned_dataframe, artifacts=None, fit=False):
    """One-hot encode nominal features and scale numeric features.

    Ordinal and binary features are already numeric before this step.
    Nominal features such as home ownership and loan intent are one-hot encoded.
    """

    encoded = pd.get_dummies(
        cleaned_dataframe,
        columns=["person_home_ownership", "loan_intent"],
        drop_first=False
    )

    if artifacts is None:
        artifacts = {}

    if fit:
        artifacts["model_columns"] = encoded.columns.tolist()
        artifacts["scaler"] = MinMaxScaler()
        scaled_array = artifacts["scaler"].fit_transform(encoded)
    else:
        encoded = encoded.reindex(columns=artifacts["model_columns"], fill_value=0)
        scaled_array = artifacts["scaler"].transform(encoded)

    scaled = pd.DataFrame(
        scaled_array,
        columns=artifacts["model_columns"],
        index=encoded.index
    )

    return scaled, artifacts


def preprocess_for_modeling(raw_features, artifacts=None, fit=False):
    """Full preprocessing wrapper used for train, validation, and test features."""

    cleaned, artifacts = base_clean_features(raw_features, artifacts=artifacts, fit=fit)
    model_ready, artifacts = encode_and_scale_features(cleaned, artifacts=artifacts, fit=fit)

    return cleaned, model_ready, artifacts


## 6. Apply Preprocessing

The training set is fitted first.

Validation and test sets reuse the training artifacts. This is the correct pattern because the model should not learn medians, categories, or scaling ranges from validation or test data.


In [7]:
clean_train, X_train, preprocessing_artifacts = preprocess_for_modeling(
    X_train_raw,
    fit=True
)

clean_val, X_val, _ = preprocess_for_modeling(
    X_val_raw,
    artifacts=preprocessing_artifacts,
    fit=False
)

clean_test, X_test, _ = preprocess_for_modeling(
    X_test_raw,
    artifacts=preprocessing_artifacts,
    fit=False
)

print("Clean training shape:", clean_train.shape)
print("Model-ready training shape:", X_train.shape)
print("Model-ready validation shape:", X_val.shape)
print("Model-ready test shape:", X_test.shape)

print("\nCorrection checks:")
print("Max capped person_emp_length:", clean_train["person_emp_length"].max())
print("Loan grade values after ordinal encoding:", sorted(clean_train["loan_grade"].dropna().unique()))
print(
    "Skew after log transform - income:",
    round(clean_train["person_income"].skew(), 3),
    "loan amount:",
    round(clean_train["loan_amnt"].skew(), 3)
)

print("\nMissing values after cleaning:")
display(clean_train.isnull().sum().to_frame("Missing Count"))


Clean training shape: (22806, 14)
Model-ready training shape: (22806, 22)
Model-ready validation shape: (4887, 22)
Model-ready test shape: (4888, 22)

Correction checks:
Max capped person_emp_length: 100.0
Loan grade values after ordinal encoding: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7)]
Skew after log transform - income: 0.147 loan amount: -0.456

Missing values after cleaning:


,Missing Count
person_age,0
person_income,0
person_home_ownership,0
person_emp_length,0
loan_intent,0
loan_grade,0
loan_amnt,0
loan_int_rate,0
loan_percent_income,0
cb_person_default_on_file,0


## 7. Save Cleaned Datasets

These CSV files are useful deliverables because they show what the model actually used after preprocessing.


In [8]:
# Add the target column back when saving cleaned split files.
clean_train_with_target = clean_train.copy()
clean_train_with_target[target_column] = y_train.values

clean_val_with_target = clean_val.copy()
clean_val_with_target[target_column] = y_val.values

clean_test_with_target = clean_test.copy()
clean_test_with_target[target_column] = y_test.values

clean_train_with_target.to_csv("08_Clean_Training_Dataset.csv", index=False)
clean_val_with_target.to_csv("09_Clean_Validation_Dataset.csv", index=False)
clean_test_with_target.to_csv("10_Clean_Test_Dataset.csv", index=False)

final_model_ready = pd.concat(
    [clean_train_with_target, clean_val_with_target, clean_test_with_target],
    axis=0
)

final_model_ready.to_csv("11_Final_Clean_Model_Ready_Dataset.csv", index=False)

print("Saved cleaned datasets:")
print("- 08_Clean_Training_Dataset.csv")
print("- 09_Clean_Validation_Dataset.csv")
print("- 10_Clean_Test_Dataset.csv")
print("- 11_Final_Clean_Model_Ready_Dataset.csv")


Saved cleaned datasets:
- 08_Clean_Training_Dataset.csv
- 09_Clean_Validation_Dataset.csv
- 10_Clean_Test_Dataset.csv
- 11_Final_Clean_Model_Ready_Dataset.csv


## 8. Define Models

The final rubric requires the following classifiers:

- Logistic Regression
- Decision Tree Classifier
- Random Forest Classifier
- Gradient Boosting Classifier
- K-Nearest Neighbors Classifier
- Support Vector Classifier

For the SVC requirement, this notebook uses `LinearSVC`. It trains fast and is stable for a support-vector classifier.


In [9]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE, max_depth=8),
    "Random Forest": RandomForestClassifier(
        n_estimators=150,
        random_state=RANDOM_STATE,
        max_depth=12,
        n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=7),
    "Support Vector Classifier": LinearSVC(random_state=RANDOM_STATE, max_iter=5000),
}

print("Models defined:")
for model_name in models:
    print("-", model_name)


Models defined:
- Logistic Regression
- Decision Tree
- Random Forest
- Gradient Boosting
- K-Nearest Neighbors
- Support Vector Classifier


## 9. Evaluation Helper

The evaluation function produces the required classification metrics:

- Accuracy
- Precision
- Recall
- F1-Score
- ROC-AUC

ROC-AUC uses predicted probabilities when available. For `LinearSVC`, it uses the decision score instead.


In [10]:
def get_model_scores(model, features):
    """Return model confidence scores for ROC-AUC and ensemble averaging.

    Classifiers expose confidence differently:
    - predict_proba gives probability estimates
    - decision_function gives distance from the decision boundary
    - predict gives only hard class predictions as a fallback
    """

    if hasattr(model, "predict_proba"):
        return model.predict_proba(features)[:, 1]

    if hasattr(model, "decision_function"):
        scores = model.decision_function(features)

        # Convert decision scores to a 0-1 range for ensemble compatibility.
        score_min = np.min(scores)
        score_max = np.max(scores)

        if score_max == score_min:
            return np.full_like(scores, 0.5, dtype=float)

        return (scores - score_min) / (score_max - score_min)

    return model.predict(features)


def evaluate_classifier(model, features, labels, dataset_name, model_name):
    """Evaluate one classifier and return a metrics dictionary."""

    predictions = model.predict(features)
    scores = get_model_scores(model, features)

    return {
        "Model": model_name,
        "Dataset": dataset_name,
        "Accuracy": accuracy_score(labels, predictions),
        "Precision": precision_score(labels, predictions, zero_division=0),
        "Recall": recall_score(labels, predictions, zero_division=0),
        "F1-Score": f1_score(labels, predictions, zero_division=0),
        "ROC-AUC": roc_auc_score(labels, scores),
    }


## 10. Train and Validate All Models

Each required model is trained on the 70% training split.

The validation set is then used to compare model performance before final testing.


In [11]:
trained_models = {}
results = []

for model_name, model in models.items():
    print(f"Training: {model_name}")
    model.fit(X_train, y_train)

    trained_models[model_name] = model

    results.append(
        evaluate_classifier(model, X_val, y_val, "Validation", model_name)
    )

validation_results = pd.DataFrame(results).sort_values(
    by="ROC-AUC",
    ascending=False
)

display(validation_results.round(4))


Training: Logistic Regression
Training: Decision Tree
Training: Random Forest
Training: Gradient Boosting
Training: K-Nearest Neighbors
Training: Support Vector Classifier


,Model,Dataset,Accuracy,Precision,Recall,F1-Score,ROC-AUC
2,Random Forest,Validation,0.9329,0.9855,0.7026,0.8204,0.9334
3,Gradient Boosting,Validation,0.9269,0.9426,0.7083,0.8088,0.9327
1,Decision Tree,Validation,0.9304,0.9678,0.7045,0.8154,0.9170
5,Support Vector Classifier,Validation,0.8615,0.7791,0.5094,0.6160,0.8812
0,Logistic Regression,Validation,0.8623,0.7673,0.5291,0.6263,0.8778
4,K-Nearest Neighbors,Validation,0.8999,0.8979,0.6107,0.7270,0.8723


## 11. Final Test Evaluation

After model comparison on validation data, the same models are evaluated on the isolated final test set.

This is the performance estimate that best represents how the models may behave on unseen data.


In [12]:
test_results = []

for model_name, model in trained_models.items():
    test_results.append(
        evaluate_classifier(model, X_test, y_test, "Test", model_name)
    )

test_results = pd.DataFrame(test_results).sort_values(
    by="ROC-AUC",
    ascending=False
)

display(test_results.round(4))

model_comparison_table = pd.concat(
    [validation_results, test_results],
    axis=0
)

model_comparison_table.to_csv("12_Model_Comparison_Table.csv", index=False)

print("Saved model comparison table: 12_Model_Comparison_Table.csv")


,Model,Dataset,Accuracy,Precision,Recall,F1-Score,ROC-AUC
3,Gradient Boosting,Test,0.9315,0.9453,0.7282,0.8227,0.9317
2,Random Forest,Test,0.9349,0.9807,0.7160,0.8277,0.9309
1,Decision Tree,Test,0.9315,0.9564,0.7188,0.8208,0.9108
5,Support Vector Classifier,Test,0.8631,0.7803,0.5192,0.6235,0.8805
0,Logistic Regression,Test,0.8629,0.7672,0.5342,0.6298,0.8768
4,K-Nearest Neighbors,Test,0.8938,0.8673,0.6064,0.7137,0.8738


Saved model comparison table: 12_Model_Comparison_Table.csv


## 12. Voting Ensemble

The voting ensemble combines the top three validation models.

This helps reduce dependence on a single model and often improves stability.


In [13]:
top_three_model_names = validation_results.head(3)["Model"].tolist()

print("Top three validation models selected for ensemble:")
for name in top_three_model_names:
    print("-", name)

# VotingClassifier requires estimators to be passed as (name, model) tuples.
# Hard voting is used because LinearSVC does not expose predict_proba by default.
voting_estimators = [
    (name.replace(" ", "_").lower(), trained_models[name])
    for name in top_three_model_names
]

voting_ensemble = VotingClassifier(
    estimators=voting_estimators,
    voting="hard"
)

voting_ensemble.fit(X_train, y_train)

voting_validation_metrics = evaluate_classifier(
    voting_ensemble,
    X_val,
    y_val,
    "Validation",
    "Voting Ensemble"
)

voting_test_metrics = evaluate_classifier(
    voting_ensemble,
    X_test,
    y_test,
    "Test",
    "Voting Ensemble"
)

display(pd.DataFrame([voting_validation_metrics, voting_test_metrics]).round(4))


Top three validation models selected for ensemble:
- Random Forest
- Gradient Boosting
- Decision Tree


,Model,Dataset,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,Voting Ensemble,Validation,0.9325,0.9817,0.7036,0.8197,0.8500
1,Voting Ensemble,Test,0.9351,0.9771,0.7198,0.8289,0.8575


## 13. Bayesian-Style Weighted Ensemble

The Bayesian ensemble uses validation ROC-AUC scores to weight the top three models.

Presentation explanation:

- Better validation models receive stronger weights.
- Weights are normalized so they sum to 1.
- Final predictions are based on weighted average model confidence.

This is a practical Bayesian-style model averaging approach for the class assignment.


In [14]:
def softmax(values):
    """Convert numeric values into normalized weights that sum to 1."""
    values = np.array(values, dtype=float)
    shifted = values - values.max()
    exp_values = np.exp(shifted)
    return exp_values / exp_values.sum()


top_three_validation = validation_results.head(3).copy()
bayesian_weights = softmax(top_three_validation["ROC-AUC"].values)

weight_table = pd.DataFrame({
    "Model": top_three_validation["Model"].values,
    "Validation ROC-AUC": top_three_validation["ROC-AUC"].values,
    "Bayesian Ensemble Weight": bayesian_weights,
})

display(weight_table.round(4))


def bayesian_weighted_predictions(features, threshold=0.50):
    """Create weighted ensemble predictions from top validation models."""

    weighted_scores = np.zeros(features.shape[0])

    for model_name, weight in zip(top_three_validation["Model"].values, bayesian_weights):
        model_scores = get_model_scores(trained_models[model_name], features)
        weighted_scores += weight * model_scores

    predictions = (weighted_scores >= threshold).astype(int)

    return predictions, weighted_scores


def evaluate_bayesian_ensemble(features, labels, dataset_name):
    """Evaluate the Bayesian-style weighted ensemble."""

    predictions, scores = bayesian_weighted_predictions(features)

    return {
        "Model": "Bayesian Weighted Ensemble",
        "Dataset": dataset_name,
        "Accuracy": accuracy_score(labels, predictions),
        "Precision": precision_score(labels, predictions, zero_division=0),
        "Recall": recall_score(labels, predictions, zero_division=0),
        "F1-Score": f1_score(labels, predictions, zero_division=0),
        "ROC-AUC": roc_auc_score(labels, scores),
    }


bayesian_validation_metrics = evaluate_bayesian_ensemble(X_val, y_val, "Validation")
bayesian_test_metrics = evaluate_bayesian_ensemble(X_test, y_test, "Test")

ensemble_comparison = pd.DataFrame([
    voting_validation_metrics,
    voting_test_metrics,
    bayesian_validation_metrics,
    bayesian_test_metrics,
])

display(ensemble_comparison.round(4))

ensemble_comparison.to_csv("13_Ensemble_Comparison_Table.csv", index=False)

ensemble_details = {
    "top_three_models": top_three_model_names,
    "bayesian_weights": weight_table.to_dict(orient="records"),
    "explanation": (
        "Bayesian-style weights were generated from validation ROC-AUC scores "
        "and used to average the top model confidence scores."
    )
}

with open("14_Ensemble_Details.json", "w") as f:
    json.dump(ensemble_details, f, indent=2)

print("Saved ensemble comparison table: 13_Ensemble_Comparison_Table.csv")
print("Saved ensemble details: 14_Ensemble_Details.json")


,Model,Validation ROC-AUC,Bayesian Ensemble Weight
0,Random Forest,0.9334,0.3352
1,Gradient Boosting,0.9327,0.3350
2,Decision Tree,0.9170,0.3298


,Model,Dataset,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,Voting Ensemble,Validation,0.9325,0.9817,0.7036,0.8197,0.8500
1,Voting Ensemble,Test,0.9351,0.9771,0.7198,0.8289,0.8575
2,Bayesian Weighted Ensemble,Validation,0.9329,0.9830,0.7045,0.8208,0.9330
3,Bayesian Weighted Ensemble,Test,0.9339,0.9721,0.7179,0.8259,0.9316


Saved ensemble comparison table: 13_Ensemble_Comparison_Table.csv
Saved ensemble details: 14_Ensemble_Details.json


## 14. Final Explanation

The biggest corrections and decisions were:

1. `person_emp_length` was capped at 100 instead of scaled.
2. `loan_grade` was ordinal encoded from A=1 through G=7.
3. Skew was addressed for both `person_income` and `loan_amnt`.
4. The original 30% holdout was split into 15% validation and 15% final testing.
5. The validation set was used for model comparison.
6. The final test set remained isolated until final evaluation.

The final workflow shows a complete classification modeling process with clean preprocessing, multiple models, metric comparison, and ensemble evaluation.


In [ ]:
print("Notebook complete.")

print("\nTalking points:")
print("1. This is a binary classification problem predicting loan_status.")
print("2. The train/validation/test split is 70/15/15.")
print("3. The 30% holdout was split into validation and final test sets.")
print("4. Employment length was capped at 100 to correct the 123-year outlier.")
print("5. Loan grade was ordinal encoded because the grades have a risk order.")
print("6. Income and loan amount skew were corrected with log transforms.")
print("7. All models and ensemble methods were trained and compared.")


Notebook complete.

Presentation talking points:
1. This is a binary classification problem predicting loan_status.
2. The train/validation/test split is 70/15/15.
3. The 30% holdout was split into validation and final test sets.
4. Employment length was capped at 100 to correct the 123-year outlier.
5. Loan grade was ordinal encoded because the grades have a risk order.
6. Income and loan amount skew were corrected with log transforms.
7. All required models and ensemble methods were trained and compared.
